# Experiment 9 (pilot) — Does a law-forced cycle survive the story wrapper?

**Why this notebook exists.** The [experiment-9 proposal](09-law-forced-manifold-proposal.md)
adapts Goodfire's manifold steering (Wurgaft et al., arXiv:2605.05115) to our setting. That
method needs a *behavior manifold*: a next-token distribution over a small concept set $Z$ with a
known metric, on a task the model is actually competent at. Everything downstream — the
$M_h \leftrightarrow M_y$ isometry, manifold-vs-linear steering, the pullback — is computed from
that distribution. If the model is not competent, or if its errors are unstructured, there is no
$M_y$ to fit and the design collapses.

This pilot is the competence gate (**H1** in the proposal). It is deliberately small: no spline
fitting, no steering, no intervention. One forward pass per prompt, read one token position.

**The task.** We state a finite magma whose habit forces a $k$-cycle: *combining anything with*
$b$ *yields the element listed immediately after* $b$. A chain of $n$ applications therefore
lands at $\mathrm{succ}^n(\text{start})$, so the answer domain is a cycle with a known metric.
The same magma and the same chains are rendered in six surface forms — bare notation, literal
prose, and four themed stories — so surface form is the only thing that varies.

**Why paint colours would carry a cycle at all.** They would not, from pretraining: nothing says
ochre follows crimson. The cycle is imposed by the habit stated in the prompt, the way Goodfire's
in-context-graph experiment assigns arbitrary nouns to grid nodes. That is the point — we are
testing whether an *explicitly stated law* induces geometry, not whether the model memorised one.

**Questions this pilot answers.**

1. Can Qwen3-4B compute in this magma at all, per surface form?
2. Does it put its probability mass on the concept set, or leak it off-concept?
3. When it is wrong, does the mass land on *cycle neighbours*? (This is what makes $M_y$
   non-degenerate — Goodfire's weekday distributions peak on the answer and spill onto adjacent
   days.)
4. Preview only: do behaviour and activation centroids already recover the cyclic ordering,
   before any manifold is fitted?

**Kill criteria.** Any of these stops the full experiment as designed:

- Accuracy at or near chance ($1/k$) in *every* form, including bare notation → the model cannot
  do the task; the geometry question is unaskable.
- Concept mass near zero → the readout position is wrong, or the model refuses the format. Fix
  the harness before concluding anything.
- Errors uniform over non-answers in every form → output distributions carry no metric structure,
  so $M_y$ is a set of simplex corners with no curve through them.

A **partial** pass is the interesting case and is *not* a kill: competence in bare notation with
collapse under narrative is precisely hypothesis H3, and would be the headline result.

**Runtime.** Colab GPU, ~1200 short prompts, one forward pass each. Minutes, not hours.

In [ ]:
# ------------------------------ Configuration ------------------------------

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # same pin as experiments 01, 03, 08

MODEL_NAME = "Qwen/Qwen3-4B"
SEED = 0

# The magma: CYCLE_K elements in a stated cyclic order, a * b = successor(b).
# Capped at 6 because that is how many names a theme palette carries. k=5 leaves
# one spare name per theme in case of a tokeniser collision; k=6 gives a finer
# distance scale (distances 1..3 instead of 1..2) with no slack.
CYCLE_K = 5

# Chain lengths. Mirrors Goodfire's (entity, increment) enumeration: every answer
# value is reached once per chain length, so prompt length is balanced across
# answer values rather than held constant within one.
CHAIN_LENGTHS = [1, 2, 3, 4, 5]

# Distractor draws per (start, length). The habit makes the *first* argument
# irrelevant to the result, so these vary the prompt without moving the answer:
# within-concept variation for centroids, and a free test of whether the model
# correctly ignores them.
DISTRACTOR_DRAWS = 8

FORMS = ["symbolic", "literal", "paint", "tea", "graft", "signal"]

# Residual-stream read points (hidden_states indices), for the activation preview
# only. 18/24 are experiment 3's law-retrieval peak; 28 is nearer the late layer
# Goodfire used (28 of 32).
LAYERS = [12, 18, 24, 28]
PRIMARY_LAYER = 24

# Full-vocabulary logits are materialised for the whole batch, and that — not the
# weights — is the memory driver here. Experiment 3 OOM'd on a similar pass; drop
# this to 4 before touching anything else if you hit it.
BATCH_SIZE = 8

# Smoke test: validates rendering, tokenisation and the readout on two forms.
QUICK_TEST = False
if QUICK_TEST:
    DISTRACTOR_DRAWS = 2
    FORMS = ["symbolic", "paint"]
    LAYERS = [24]

In [ ]:
# ------------------------- Environment and outputs -------------------------

%pip install -q -U "transformers>=4.51" accelerate matplotlib

import hashlib
import itertools
import json
import random
import subprocess
import sys
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

# Only the theme definitions and the equation parser are reused: the orbit task
# needs a new renderer, since storyform renders implication questions rather than
# evaluation chains.
from storyform import THEMES, Var, parse_equation, variables_in_order

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

assert CYCLE_K <= 6, "theme palettes carry six names"

OUT_DIR = Path("exp09-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp09-pilot")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())

## The magma, and the laws it actually satisfies

The habit is $a \ast b = \mathrm{succ}(b)$ on $k$ elements in a stated cyclic order. Two claims
matter for the framing, and neither is asserted — both are checked by brute force over every
assignment, using the project's own equation parser so the laws are written in ETP notation:

- **`x ◇ y = z ◇ y`** — the first argument is irrelevant. This is what makes the distractor
  arguments provably inert, and it is an ordinary ETP-shaped law (three variables, one operation
  per side).
- **the $k$-fold closure law** — applying the operation $k$ times returns you to where you
  started. This is what closes the orbit into a cycle rather than a line, and therefore what
  makes the concept domain's metric cyclic.

A law that should *fail* (`x ◇ y = y`, the operation doing nothing) is checked too, so the
verifier is shown to discriminate rather than rubber-stamp.

One caveat to carry into the write-up: these laws are *satisfied by* this magma, they do not
uniquely axiomatise it, and the prompt states the magma operationally rather than stating the law
and letting the model find a model of it. Which of the two the full experiment should use is an
open design question in the proposal.

In [ ]:
# --------------- The magma, and mechanical verification of its laws ---------------

# Cayley table for a * b = successor(b) on Z_k.
TABLE = [[(b + 1) % CYCLE_K for b in range(CYCLE_K)] for _ in range(CYCLE_K)]


def eval_term(term, env):
    if isinstance(term, Var):
        return env[term.name]
    return TABLE[eval_term(term.left, env)][eval_term(term.right, env)]


def law_holds(equation_text):
    """Exhaustive check that a law holds in this magma, over every assignment."""
    lhs, rhs = parse_equation(equation_text)
    names = variables_in_order(lhs, rhs)
    for values in itertools.product(range(CYCLE_K), repeat=len(names)):
        env = dict(zip(names, values))
        if eval_term(lhs, env) != eval_term(rhs, env):
            return False
    return True


def closure_law(k):
    """succ^k = identity, as a right-nested term: a1 ◇ (a2 ◇ (... ◇ (ak ◇ t))) = t."""
    text = "t"
    for i in range(k):
        text = f"a{i + 1} ◇ ({text})" if i else f"a{i + 1} ◇ t"
    return f"{text} = t"


FIRST_ARG_IRRELEVANT = "x ◇ y = z ◇ y"
CLOSURE = closure_law(CYCLE_K)
SHOULD_FAIL = "x ◇ y = y"

assert law_holds(FIRST_ARG_IRRELEVANT), "first argument is not inert — distractors would matter"
assert law_holds(CLOSURE), "the orbit does not close after k steps"
assert not law_holds(SHOULD_FAIL), "verifier accepts a false law"

# The orbit is a single k-cycle, not a union of shorter ones.
orbit = [0]
while len(orbit) < CYCLE_K:
    orbit.append(TABLE[0][orbit[-1]])
assert sorted(orbit) == list(range(CYCLE_K)), "successor does not generate one full cycle"

print(f"magma on {CYCLE_K} elements, a * b = succ(b)")
print(f"  holds:  {FIRST_ARG_IRRELEVANT}")
print(f"  holds:  {CLOSURE}")
print(f"  fails:  {SHOULD_FAIL}   (negative control)")


def cyclic_distance(i, j):
    d = abs(i - j) % CYCLE_K
    return min(d, CYCLE_K - d)


TRUE_DISTANCE = np.array(
    [[cyclic_distance(i, j) for j in range(CYCLE_K)] for i in range(CYCLE_K)], dtype=float
)
print(f"\ndistinct cyclic distances: {sorted(set(TRUE_DISTANCE[np.triu_indices(CYCLE_K, 1)]))}")

## Element names, chosen so the readout can tell them apart

The behaviour distribution is read from a single next-token position, so two elements whose names
share a first token are indistinguishable there and would silently merge two concepts into one.
The tokeniser is therefore loaded *before* the dataset is built (cheap, no GPU) and the palette is
chosen against it: walk each theme's six names and keep the first $k$ whose spelling variants do
not overlap anything already taken.

Element names differ across arms by design — `symbolic` and `literal` use `A`–`E`, the themes use
their own palettes. Every metric below (accuracy, spill, distance matrices) is invariant to
relabelling; what has to stay aligned across arms is the *cycle position*, not the word.

In [ ]:
# ----------------- Tokeniser, and a collision-free name per position -----------------

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def variant_token_ids(name):
    """First-token ids for a concept's spellings; Goodfire aggregate variants likewise."""
    ids = set()
    capitalized = f"{name[:1].upper()}{name[1:]}"
    # Both spacings of both casings: the element follows "**" with no space, but the
    # spaced variants stay in the set so the readout survives a prefix change.
    for spelling in (f" {name}", name, f" {capitalized}", capitalized):
        encoded = tokenizer.encode(spelling, add_special_tokens=False)
        if encoded:
            ids.add(encoded[0])
    return frozenset(ids)


def pick_names(candidates, form):
    chosen, taken = [], set()
    for name in candidates:
        ids = variant_token_ids(name)
        if not ids or (ids & taken):
            continue
        chosen.append(name)
        taken |= ids
        if len(chosen) == CYCLE_K:
            return chosen
    raise SystemExit(
        f"{form}: only {len(chosen)} of {CYCLE_K} names have distinct first tokens "
        f"among {list(candidates)}; lower CYCLE_K or edit the theme palette"
    )


ABSTRACT_NAMES = ["A", "B", "C", "D", "E", "F"]
FORM_NAMES = {
    form: pick_names(
        ABSTRACT_NAMES if form in ("symbolic", "literal") else THEMES[form].palette, form
    )
    for form in FORMS
}
TOKEN_IDS = {form: [sorted(variant_token_ids(n)) for n in names]
             for form, names in FORM_NAMES.items()}

for form, names in FORM_NAMES.items():
    print(f"{form:>9}: {names}")

## Six surface forms of the same chain

Every instance is the same computation — start at element $s$, apply the operation $n$ times with
inert first arguments — dressed six ways:

| form | register |
|---|---|
| `symbolic` | bare notation, `v1 = D * A` |
| `literal` | abstract prose, `Value 1`, no narrative |
| `paint` `tea` `graft` `signal` | themed narrative, reusing each theme's own operation verb and result noun |

`symbolic` vs `literal` isolates notation from prose with the names held fixed; `literal` vs the
themes isolates narrative and concrete nouns. All six end with the identical assistant prefix, so
the readout position means the same thing everywhere.

In [ ]:
# ----------------------------- Renderers -----------------------------

# Identical across forms, so the readout position means the same thing everywhere.
# The trailing "**" is load-bearing: asked for one word, Qwen3 bolds it, putting
# p(" **") > 0.98 at the position after "is" and leaving every element name at ~1e-8.
# Absorbing the markdown into the prefix puts the element at the very next token.
ANSWER_PREFIX = "The answer is **"


def _listing(names):
    return ", ".join(names[:-1]) + f", and {names[-1]}"


def render_symbolic(names, start, distractors):
    lines = [
        f"Elements in cyclic order: {', '.join(names)}.",
        f"Rule: for any elements a and b, a * b = the element immediately after b in that "
        f"order, wrapping around to {names[0]} after {names[-1]}.",
        "",
        "Compute:",
    ]
    ref = names[start]
    for i, d in enumerate(distractors, 1):
        lines.append(f"  v{i} = {names[d]} * {ref}")
        ref = f"v{i}"
    lines += ["", f"What element is {ref}? Answer with one word."]
    return "\n".join(lines)


def render_literal(names, start, distractors):
    sentences = [
        f"There are {CYCLE_K} values, listed in this cyclic order: {_listing(names)}.",
        "Applying the operation to any value as its first input and any value as its second "
        "input yields the value listed immediately after the second input, wrapping around "
        f"to {names[0]} after {names[-1]}.",
    ]
    ref = f"value {names[start]}"
    for i, d in enumerate(distractors, 1):
        sentences.append(
            f"Apply the operation to value {names[d]} as its first input and {ref} as its "
            f"second input, and call the result Value {i}."
        )
        ref = f"Value {i}"
    sentences.append(f"Which value is {ref}? Answer with one word.")
    return " ".join(sentences)


def render_story(form, names, start, distractors):
    theme = THEMES[form]
    steps, ref = [], names[start]
    for i, d in enumerate(distractors, 1):
        clause = theme.op_imperative.format(a=names[d], b=ref)
        steps.append(f"{clause} and call the result {theme.result_noun} {i}")
        ref = f"{theme.result_noun} {i}"
    return "\n\n".join([
        f"{theme.intro} There are {CYCLE_K} {theme.element_plural}, in this fixed order: "
        f"{_listing(names)}.",
        f"The habit is this. Take any two {theme.element_plural} and "
        f"{theme.op_imperative.format(a='the first', b='the second')}: what comes out is always "
        f"the {theme.element_singular} listed immediately after the second one, wrapping around "
        f"to {names[0]} after {names[-1]}. {theme.closing}",
        f"One morning, {'; then '.join(steps)}.",
        f"Which {theme.element_singular} is {ref}? Answer with one word.",
    ])


def render(form, start, distractors):
    names = FORM_NAMES[form]
    if form == "symbolic":
        return render_symbolic(names, start, distractors)
    if form == "literal":
        return render_literal(names, start, distractors)
    return render_story(form, names, start, distractors)

In [ ]:
# --------------------------- Build the dataset ---------------------------

rng = random.Random(SEED)

# One instance spec per (start, chain length, distractor draw), shared by every
# form so the arms differ in wording alone.
specs = []
for start in range(CYCLE_K):
    for n in CHAIN_LENGTHS:
        for draw in range(DISTRACTOR_DRAWS):
            specs.append({
                "start": start,
                "length": n,
                "draw": draw,
                "distractors": [rng.randrange(CYCLE_K) for _ in range(n)],
                "answer": (start + n) % CYCLE_K,  # succ^n(start)
            })

records = [
    {**spec, "form": form, "text": render(form, spec["start"], spec["distractors"])}
    for form in FORMS
    for spec in specs
]

counts = np.bincount([s["answer"] for s in specs], minlength=CYCLE_K)
assert counts.min() == counts.max(), "answer values are not balanced"
print(f"{len(specs)} instances x {len(FORMS)} forms = {len(records)} prompts")
print(f"instances per answer value: {counts.tolist()}")

example = specs[2 * DISTRACTOR_DRAWS]  # start 0, a mid-length chain
for form in FORMS[:3]:
    print(f"\n{'=' * 22} {form} {'=' * 22}")
    print(render(form, example["start"], example["distractors"]))
    print(f"  [{ANSWER_PREFIX} -> {FORM_NAMES[form][example['answer']]}]")

## Readout: one token position, one distribution over the concept set

Following Goodfire's protocol: softmax over the full vocabulary at the answer position, aggregate
each concept's spelling variants into one entry, and collect everything else into a single
`other` bin. That yields a point on the simplex over $k + 1$ classes — the raw material for a
behaviour manifold.

The `other` bin is not a throwaway. If it holds most of the mass, the model is not answering in
the requested format, and every geometric statement downstream would be about noise.

Activations at the same position are cached in the same pass for the section-4 preview.

In [ ]:
# ---------------- Forward pass: concept distributions and activations ----------------

RUN_SHA = hashlib.sha256(
    json.dumps([MODEL_NAME, CYCLE_K, CHAIN_LENGTHS, DISTRACTOR_DRAWS, FORMS, LAYERS, SEED,
                ANSWER_PREFIX, [r["text"] for r in records]]).encode()).hexdigest()[:12]
CACHE = OUT_DIR / f"readout-{RUN_SHA}.pt"

if CACHE.exists():
    blob = torch.load(CACHE)
    probs, acts = blob["probs"], blob["acts"]
    print(f"loaded cached readout for {probs.shape[0]} prompts")
else:
    from transformers import AutoModelForCausalLM

    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.float32
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto")
    model.eval()
    assert max(LAYERS) <= model.config.num_hidden_layers

    def build_chat(text):
        chat = tokenizer.apply_chat_template(
            [{"role": "user", "content": text}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return chat + ANSWER_PREFIX

    chats = [build_chat(r["text"]) for r in records]
    forms = [r["form"] for r in records]

    @torch.no_grad()
    def read(chats, forms):
        tokenizer.padding_side = "right"  # keeps last-token indexing by length valid
        out_probs, out_acts = [], {L: [] for L in LAYERS}
        for i in tqdm(range(0, len(chats), BATCH_SIZE), desc="readout"):
            batch, batch_forms = chats[i:i + BATCH_SIZE], forms[i:i + BATCH_SIZE]
            encoded = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
            output = model(**encoded, output_hidden_states=True)
            lengths = encoded["attention_mask"].sum(dim=1)
            rows = torch.arange(lengths.size(0), device=lengths.device)
            last = lengths - 1
            full = torch.softmax(output.logits[rows, last].float(), dim=-1).cpu()
            block = torch.zeros(len(batch), CYCLE_K + 1)
            for b, form in enumerate(batch_forms):
                for c, ids in enumerate(TOKEN_IDS[form]):
                    block[b, c] = full[b, ids].sum()
                block[b, CYCLE_K] = (1.0 - block[b, :CYCLE_K].sum()).clamp(min=0.0)
            out_probs.append(block)
            for L in LAYERS:
                out_acts[L].append(output.hidden_states[L].float()[rows, last].cpu())
            del output
        return torch.cat(out_probs), {L: torch.cat(v) for L, v in out_acts.items()}

    probs, acts = read(chats, forms)
    torch.save({"probs": probs, "acts": acts}, CACHE)
    print(f"saved readout -> {CACHE.name}")

probs_np = probs.numpy()
form_of = np.array([r["form"] for r in records])
answer_of = np.array([r["answer"] for r in records])
length_of = np.array([r["length"] for r in records])

## Questions 1–2: is the model competent, and is the mass on-concept?

`accuracy` is argmax over the $k$ concept entries, against chance $1/k$. `concept mass` is the
probability that landed on the concept set at all. `p(answer)` is the mean probability on the
correct element — Goodfire's tasks produce sharply peaked distributions, and how peaked ours are
determines whether behaviour centroids will be distinguishable at all.

The by-length table matters for interpretation: chain length is a serial-compute axis that
depresses accuracy on its own, so a form effect must not be read off a table where the forms
differ in how far they get down the chain.

In [ ]:
# ------------------------- Competence and mass by form -------------------------

concept = probs_np[:, :CYCLE_K]
correct = concept.argmax(axis=1) == answer_of
p_answer = concept[np.arange(len(records)), answer_of]
concept_mass = concept.sum(axis=1)

print(f"chance accuracy = {1 / CYCLE_K:.1%}\n")
header = f"{'form':>9} {'accuracy':>9} {'p(answer)':>10} {'concept mass':>13}"
print(header + "\n" + "-" * len(header))
summary = {}
for form in FORMS:
    m = form_of == form
    summary[form] = {
        "accuracy": float(correct[m].mean()),
        "p_answer": float(p_answer[m].mean()),
        "concept_mass": float(concept_mass[m].mean()),
    }
    s = summary[form]
    print(f"{form:>9} {s['accuracy']:>9.1%} {s['p_answer']:>10.3f} {s['concept_mass']:>13.1%}")

print("\naccuracy by chain length")
print(f"{'form':>9} " + " ".join(f"{n:>6}" for n in CHAIN_LENGTHS))
for form in FORMS:
    cells = []
    for n in CHAIN_LENGTHS:
        m = (form_of == form) & (length_of == n)
        cells.append(f"{correct[m].mean():>6.0%}" if m.any() else f"{'-':>6}")
    print(f"{form:>9} " + " ".join(cells))

## Question 3: does the error mass land on cycle neighbours?

This is the decisive one. A behaviour manifold exists only if the output distribution knows the
*metric*, not merely the answer — Goodfire's weekday distributions concentrate the remainder on
adjacent days, and that is what makes $M_y$ a curve rather than a scatter of simplex corners.

Metric: renormalise the non-answer concept mass, then take its mean cyclic distance from the true
answer. Lower than the uniform baseline means neighbour spill. The baseline is the mean cyclic
distance under a flat distribution over the $k-1$ non-answers, which is what "errors carry no
structure" looks like.

In [ ]:
# ---------------------------- Neighbour spill ----------------------------

off = concept.copy()
off[np.arange(len(records)), answer_of] = 0.0
off_total = off.sum(axis=1)
valid = off_total > 1e-9
off_norm = np.zeros_like(off)
off_norm[valid] = off[valid] / off_total[valid, None]

dist_to_answer = TRUE_DISTANCE[answer_of]  # (N, k)
mean_error_distance = (off_norm * dist_to_answer).sum(axis=1)

uniform_baseline = float(
    np.mean([TRUE_DISTANCE[a][np.arange(CYCLE_K) != a].mean() for a in range(CYCLE_K)])
)
print(f"uniform-error baseline (no structure): {uniform_baseline:.3f}")
print("all error mass on immediate neighbours: 1.000\n")

header = f"{'form':>9} {'mean err dist':>14} {'adjacent share':>15} {'n rows':>8}"
print(header + "\n" + "-" * len(header))
for form in FORMS:
    m = (form_of == form) & valid
    adjacent = off_norm[m][dist_to_answer[m] == 1].sum() / max(m.sum(), 1)
    summary[form]["mean_error_distance"] = float(mean_error_distance[m].mean())
    summary[form]["adjacent_share"] = float(adjacent)
    print(f"{form:>9} {mean_error_distance[m].mean():>14.3f} {adjacent:>15.1%} {m.sum():>8}")

fig, ax = plt.subplots(figsize=(7.5, 4))
width = 0.8 / len(FORMS)
positions = np.arange(CYCLE_K)
for i, form in enumerate(FORMS):
    m = form_of == form
    profile = [concept[m][dist_to_answer[m] == d].mean() if (dist_to_answer[m] == d).any() else 0.0
               for d in range(CYCLE_K)]
    ax.bar(positions + i * width, profile, width, label=form)
ax.set_xticks(positions + (len(FORMS) - 1) * width / 2)
ax.set_xticklabels([f"d={d}" for d in range(CYCLE_K)])
ax.set_xlabel("cyclic distance from the true answer")
ax.set_ylabel("mean probability")
ax.set_title("Where the probability mass sits, by surface form")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# ------------------- Control: are the inert distractors really inert? -------------------

# The magma makes the first argument irrelevant (verified above), so the model
# should assign no special mass to a distractor. The last one is the risk case:
# it sits closest to the readout position and is the obvious thing to echo.

last_distractor = np.array([r["distractors"][-1] for r in records])
distractor_sets = [set(r["distractors"]) for r in records]

print(f"{'form':>9} {'p(last distractor)':>19} {'p(unmentioned)':>16} {'ratio':>7}")
print("-" * 54)
for form in FORMS:
    echo, background = [], []
    for i in np.flatnonzero(form_of == form):
        if last_distractor[i] == answer_of[i]:
            continue  # cannot separate echo from a correct answer here
        echo.append(concept[i, last_distractor[i]])
        unmentioned = [c for c in range(CYCLE_K)
                       if c != answer_of[i] and c not in distractor_sets[i]]
        if unmentioned:
            background.append(concept[i, unmentioned].mean())
    echo_mean = float(np.mean(echo)) if echo else float("nan")
    background_mean = float(np.mean(background)) if background else float("nan")
    summary[form]["distractor_echo"] = echo_mean
    summary[form]["unmentioned_mass"] = background_mean
    ratio = echo_mean / background_mean if background_mean else float("nan")
    print(f"{form:>9} {echo_mean:>19.4f} {background_mean:>16.4f} {ratio:>7.1f}x")

print("\nA ratio near 1 means the distractors are behaving as the law says they should.")
print("A large ratio means the model is echoing a nearby name rather than computing,")
print("which would inflate accuracy whenever the echo happens to be right.")

## Question 4 (preview): do centroids already recover the cycle?

Not a manifold fit — no splines here. Just the cheap precursor: average the behaviour
distributions per answer value in Hellinger coordinates ($p \mapsto \sqrt{p}$, as the paper does),
average the activations per answer value, and ask whether the resulting pairwise distances track
cyclic distance. That is a stripped-down version of the paper's isometry test.

**The null needs care.** The obvious null — correlation under random relabellings of the cycle —
is degenerate, because a $k$-cycle has $2k$ automorphisms (rotations and reflections) that leave
the ground-truth distance matrix unchanged and therefore reproduce the observed correlation
exactly. With $k=5$ that is 10 of 120 permutations, so a naive permutation test could never report
$p < 0.083$. The null below enumerates all $k!$ relabellings and **excludes the dihedral ones**, so
it asks the meaningful question: does the true cycle order beat orderings that are genuinely
different?

A high correlation on `symbolic` and a low one on the themes would be H3 — narrative deforming the
geometry — visible before anything has been fitted.

In [ ]:
# --------------------- Centroid geometry, behaviour and activations ---------------------

TRI = np.triu_indices(CYCLE_K, 1)

# Relabellings that are symmetries of the cycle: i -> (r ± i) mod k. These leave
# TRUE_DISTANCE invariant, so they reproduce the observed correlation and must be
# kept out of the null.
DIHEDRAL = {tuple(int(v) for v in (r + s * np.arange(CYCLE_K)) % CYCLE_K)
            for r in range(CYCLE_K) for s in (1, -1)}
NON_SYMMETRIC = [np.array(p) for p in itertools.permutations(range(CYCLE_K))
                 if p not in DIHEDRAL]
print(f"null: {len(NON_SYMMETRIC)} non-symmetric relabellings "
      f"({len(DIHEDRAL)} dihedral ones excluded)\n")


def pairwise(centroids):
    diff = centroids[:, None, :] - centroids[None, :, :]
    return np.linalg.norm(diff, axis=-1)


def cycle_correlation(distance):
    return float(np.corrcoef(distance[TRI], TRUE_DISTANCE[TRI])[0, 1])


def null_p95(distance):
    draws = [np.corrcoef(distance[np.ix_(p, p)][TRI], TRUE_DISTANCE[TRI])[0, 1]
             for p in NON_SYMMETRIC]
    return float(np.percentile(draws, 95))


def centroids_for(mask, values):
    return np.stack([values[mask & (answer_of == a)].mean(axis=0) for a in range(CYCLE_K)])


hellinger = np.sqrt(probs_np)  # linearises the simplex, per the paper
activations = acts[PRIMARY_LAYER].numpy()

print(f"cycle-distance correlation, activations at layer {PRIMARY_LAYER}\n")
header = f"{'form':>9} {'behaviour r':>12} {'null p95':>9} {'activation r':>13} {'null p95':>9}"
print(header + "\n" + "-" * len(header))
geometry = {}
for form in FORMS:
    m = form_of == form
    d_behaviour = pairwise(centroids_for(m, hellinger))
    d_activation = pairwise(centroids_for(m, activations))
    geometry[form] = {
        "behaviour_r": cycle_correlation(d_behaviour),
        "behaviour_null_p95": null_p95(d_behaviour),
        "activation_r": cycle_correlation(d_activation),
        "activation_null_p95": null_p95(d_activation),
    }
    g = geometry[form]
    print(f"{form:>9} {g['behaviour_r']:>12.2f} {g['behaviour_null_p95']:>9.2f} "
          f"{g['activation_r']:>13.2f} {g['activation_null_p95']:>9.2f}")

summary_path = OUT_DIR / f"summary-{RUN_SHA}.json"
summary_path.write_text(json.dumps(
    {"config": {"model": MODEL_NAME, "k": CYCLE_K, "layer": PRIMARY_LAYER,
                "chain_lengths": CHAIN_LENGTHS, "draws": DISTRACTOR_DRAWS,
                "names": FORM_NAMES},
     "competence": summary, "geometry": geometry}, indent=2) + "\n")
print(f"\nwrote {summary_path.name}")

In [ ]:
# ------------------------ Centroid scatter, cycle order drawn in ------------------------

def project(centroids):
    centred = centroids - centroids.mean(axis=0)
    u, s, _ = np.linalg.svd(centred, full_matrices=False)
    return u[:, :2] * s[:2]


panels = [(hellinger, "behaviour"), (activations, f"activation L{PRIMARY_LAYER}")]
fig, axes = plt.subplots(2, len(FORMS), figsize=(2.6 * len(FORMS), 5.6), squeeze=False)
for col, form in enumerate(FORMS):
    m = form_of == form
    for row, (values, label) in enumerate(panels):
        xy = project(centroids_for(m, values))
        ax = axes[row][col]
        loop = np.vstack([xy, xy[:1]])  # close the path in ground-truth cycle order
        ax.plot(loop[:, 0], loop[:, 1], "-", color="0.75", zorder=1)
        ax.scatter(xy[:, 0], xy[:, 1], c=range(CYCLE_K), cmap="twilight", zorder=2)
        for i, (x, y) in enumerate(xy):
            ax.annotate(str(i), (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(form, fontsize=10)
        if col == 0:
            ax.set_ylabel(label, fontsize=9)
fig.suptitle("Centroids per answer value; grey path is the ground-truth cycle order", fontsize=11)
fig.tight_layout()
plt.show()

## How to read this

Decide in this order — later rows only mean something if the earlier ones pass.

1. **Concept mass well below ~50% anywhere.** The harness is wrong, not the model. Check the
   assistant prefix, the chat template, and the variant token ids before reading anything else.
2. **Accuracy at chance in every form, including `symbolic`.** Kill. The model cannot compute in
   the magma, so there is no behaviour for geometry to be about. The remedies are a smaller $k$,
   shorter chains, or a larger model — not a redesign of the manifold analysis.
3. **Mean error distance at the uniform baseline in every form.** Kill as designed. The output
   distribution knows the answer but not the metric, so Goodfire's isometry has nothing to attach
   to. Worth reporting: it is a clean statement about where their method stops applying.
4. **Competent with neighbour spill in `symbolic`/`literal`, degrading across themes.** Proceed;
   this is the result the full experiment is built to quantify. Note *how* it degrades — losing
   accuracy and losing geometry are different failures, and the centroid correlation separates
   them from the accuracy table.
5. **Competent with neighbour spill everywhere.** Proceed to the full design. The question becomes
   whether the manifold's *shape* differs across forms even where accuracy does not.

Three caveats to carry into the write-up.

- **Length confound.** Themed prompts are much longer than `symbolic` ones, so any form gap is
  length-confounded until the full experiment adds a length-matched control — the same caution
  experiment 0 raised about the $r = 0.93$ length/complexity correlation.
- **Coarse distance scale.** At $k = 5$ there are only two distinct cyclic distances, so the
  correlation statistic is close to a two-group comparison. `CYCLE_K = 6` gives three and is the
  natural robustness check, at the cost of using every palette name with no spare.
- **Distractors are inert by construction**, verified against the magma at the top of the
  notebook. If accuracy tracks the distractor values at all, something in the rendering is
  leaking, and that should be chased before the numbers are believed.